# Topic Analysis: ECHOE

Let's classify ECHOE documents by document topic vector. This can only be done effectively if we either create a stopword list that is considerably better than that included in CLTK, or else make use of the models' built-in arguments for excluding the most and least frequent terms, by specifying e.g.

```python
dictionary.filter_extremes(no_below=20, no_above=0.5)
```

to filter out terms that occur in fewer than 20 documents or in more than half the documents. However, since this is a matter of trial and error and we happen to know that the 350 most frequent terms make for a good stopword list in the case of ECHOE, we may as well filter those out ourselves.

We'll use NLTK's `PlaintextCorpusReader` for loading the corpus, and we'll assume you have previously run [the demo notebook on stripping identifiers from a plaintext corpus](https://github.com/langeslag/ehtc/blob/main/demo/stripping_strings.ipynb) from a subfolder of the repository root, so you have a bare, normalized plaintext corpus in `../corpora/echoe-bare`:

In [1]:
from pathlib import Path
from collections import Counter
from nltk.corpus import PlaintextCorpusReader
from copy import deepcopy
root = str(Path.cwd().parent / 'corpora' / 'echoe-bare')
echoe = PlaintextCorpusReader(root, '.*')
fdist = Counter(echoe.words())
stopwords = [k for k,v in fdist.most_common(350)]

Now we are ready to create our model:

In [2]:
from pprint import pprint
from gensim import corpora
from gensim.models import LsiModel
from gensim.models import LdaModel
import pyLDAvis.gensim_models
from nltk.tokenize import word_tokenize
corpus = []
for doc in echoe.fileids():
    tokens = echoe.words(doc)
    stopped = [i for i in tokens if not i in stopwords]
    corpus.append(stopped)
dictionary = corpora.Dictionary(corpus)
dtmatrix = [dictionary.doc2bow(doc) for doc in corpus]

We’ll try out two different topic models, Latent Semantic Analysis and Latent Dirichlet Allocation:

In [3]:
lsa = LsiModel(dtmatrix, num_topics=5, id2word=dictionary)
ldia = LdaModel(dtmatrix, num_topics=5, id2word=dictionary)

/home/paul/.pyenv/versions/3.9.18/envs/nlp/lib/python3.9/site-packages/gensim/models/lsimodel.py:963: DeprecationWarning: `scipy.sparse.sparsetools.csc_matvecs` is deprecated along with the `scipy.sparse.sparsetools` namespace. `scipy.sparse.sparsetools.csc_matvecs` will be removed in SciPy 1.14.0, and the `scipy.sparse.sparsetools` namespace will be removed in SciPy 2.0.0.
  sparsetools.csc_matvecs(


You can now inspect the topics and their composition as follows:

In [4]:
pprint(lsa.print_topics())
pprint(ldia.print_topics())

[(0,
  '0.101*"gelamp" + 0.089*"gehyrde" + 0.077*"ferde" + 0.076*"eadigan" + '
  '0.075*"byrig" + 0.073*"comon" + 0.073*"marian" + 0.072*"geond" + '
  '0.072*"nama" + 0.071*"pilatus"'),
 (1,
  '-0.345*"dauid" + -0.195*"heoræ" + -0.165*"moyses" + -0.139*"bead" + '
  '-0.130*"sonæ" + -0.128*"ðus" + -0.127*"all" + -0.126*"comen" + '
  '-0.124*"iseah" + -0.121*"ðare"'),
 (2,
  '-0.253*"marian" + -0.227*"apostolas" + -0.206*"ðæm" + -0.167*"cweðende" + '
  '-0.142*"dauid" + 0.123*"byrig" + -0.120*"ond" + 0.119*"gelamp" + '
  '-0.114*"bære" + -0.108*"cweþende"'),
 (3,
  '0.249*"pilatus" + -0.176*"marian" + -0.158*"ðæm" + 0.153*"iudas" + '
  '-0.148*"cweðende" + -0.141*"apostolas" + 0.115*"wiste" + 0.114*"eower" + '
  '-0.105*"gehyrde" + -0.102*"gelamp"'),
 (4,
  '-0.183*"pilatus" + -0.146*"guðlac" + -0.132*"gelamp" + -0.118*"weres" + '
  '-0.113*"eadigan" + -0.094*"ferde" + -0.091*"gefylled" + -0.086*"comon" + '
  '-0.082*"andswarode" + -0.081*"iudea"')]
[(0,
  '0.001*"scealt" + 0.001*"sende"

But you may find it easier to visualize LDiA data as follows (hover the circles to see the ranking of relevant terms):

In [5]:
vis = pyLDAvis.gensim_models.prepare(ldia, dtmatrix, dictionary)
pyLDAvis.enable_notebook()
pyLDAvis.display(vis)

So now we have terms grouped into topics, but where do individual documents fit in?

Firstly, we can take any member of the `dtmatrix` vector list and look up its topic scores in the model as follows:

In [6]:
lsa[dtmatrix[6]]

[(0, 9.793558628041508),
 (1, 3.237413412749056),
 (2, -5.253309790571901),
 (3, -1.4072955783160828),
 (4, 6.162202309461851)]

In [7]:
ldia[dtmatrix[6]]

[(3, 0.7816532), (4, 0.21127051)]

Or we can do the same for a range of documents:

In [8]:
counter = 1
for doc in dtmatrix[:10]:
    this_ldia = ldia[doc]
    sorted_result = sorted(this_ldia, key=lambda x: x[1], reverse=True)
    print(f'Closest match for document {counter} is topic {sorted_result[0][0]} with a score of {round(float(sorted_result[0][1]), 3)}.')
    counter += 1

Closest match for document 1 is topic 0 with a score of 0.767.
Closest match for document 2 is topic 2 with a score of 0.744.
Closest match for document 3 is topic 1 with a score of 0.745.
Closest match for document 4 is topic 1 with a score of 0.506.
Closest match for document 5 is topic 3 with a score of 0.536.
Closest match for document 6 is topic 3 with a score of 0.847.
Closest match for document 7 is topic 3 with a score of 0.773.
Closest match for document 8 is topic 1 with a score of 0.751.
Closest match for document 9 is topic 1 with a score of 0.998.
Closest match for document 10 is topic 0 with a score of 0.429.


Those document indices may not mean much to us, but we can reconstruct what they refer to because they follow the alphabetical order of files in the document folder.

Along the same lines, we can feed the model a new bag of words (or, if we can't make sense of the above indices, feed it an old bag of words anew) and ask it to identify the best topic match. This requires us to convert the document's list of tokens into a bag-of-words model using the same vocabulary used for determining the topic distribution above (i.e. `dictionary`):

In [9]:
new_docs = dict()
new_docs['judgement_day'] = dictionary.doc2bow(echoe.words('068.08.txt'))
new_docs['st_martin'] = dictionary.doc2bow(echoe.words('394.20.txt'))

In [10]:
for k,v in new_docs.items():
    this_lsa = lsa[v]
    sorted_result = sorted(this_lsa, key=lambda x: x[1], reverse=True)
    print(f'Closest LSA match for {k} is topic {sorted_result[0][0]} with a score of {sorted_result[0][1]}.')

Closest LSA match for judgement_day is topic 0 with a score of 6.524790333851905.
Closest LSA match for st_martin is topic 0 with a score of 15.649891245026277.


In [11]:
for k,v in new_docs.items():
    this_ldia = ldia[v]
    sorted_result = sorted(this_ldia, key=lambda x: x[1], reverse=True)
    print(f'Closest LDiA match for {k} is topic {sorted_result[0][0]} with a score of {sorted_result[0][1]}.')

Closest LDiA match for judgement_day is topic 3 with a score of 0.9467623233795166.
Closest LDiA match for st_martin is topic 3 with a score of 0.9679682850837708.


So what if we want to group all of our documents by topic? Wasn't that the point of all this?

The models actually don't make those groupings readily available, but we can do the work ourselves:

In [12]:
all_topics = ldia.print_topics()
docs_per_topic = [[] for _ in all_topics]
for doc_id, doc_bow in enumerate(dtmatrix):
    doc_topics = ldia.get_document_topics(doc_bow)
    for topic_id, score in doc_topics:
        docs_per_topic[topic_id].append((doc_id, score))

docs_per_topic_by_score = deepcopy(docs_per_topic)
for doc_list in docs_per_topic_by_score:
    doc_list.sort(key=lambda id_and_score: id_and_score[1], reverse=True)

In [13]:
print(docs_per_topic_by_score[0])

[(300, 0.99943566), (328, 0.99878174), (18, 0.9987783), (99, 0.9987517), (259, 0.9985587), (53, 0.9985272), (138, 0.9982087), (280, 0.9982065), (147, 0.9981562), (156, 0.99702775), (38, 0.99648756), (219, 0.99366325), (335, 0.99333173), (25, 0.9924718), (334, 0.99224705), (51, 0.991402), (72, 0.99106115), (309, 0.98851806), (125, 0.98659396), (11, 0.9840988), (296, 0.9824987), (229, 0.97786176), (90, 0.9767001), (106, 0.9729422), (157, 0.97153795), (126, 0.96719617), (144, 0.9638757), (250, 0.95893735), (96, 0.9469234), (98, 0.94492316), (61, 0.9369398), (186, 0.9096136), (247, 0.89479405), (333, 0.89135194), (240, 0.88611484), (193, 0.8837938), (57, 0.8583619), (242, 0.8516868), (139, 0.7989572), (0, 0.7859982), (185, 0.77881616), (122, 0.71245515), (124, 0.706266), (258, 0.6856361), (71, 0.66930896), (137, 0.6560763), (161, 0.63962406), (35, 0.63002264), (123, 0.57151467), (256, 0.5647938), (28, 0.5645774), (302, 0.5619289), (180, 0.5555242), (113, 0.554898), (100, 0.5175957), (78, 0

As a final step, we'll want to associate these documents with more memorable identifiers. Technically we have access to descriptive titles in ECHOE's XML headers, but we haven't cloned the XML repository, so let's make do with the filenames; we can then manually inspect the members of a topic for thematic similarity.

In [14]:
topic_groups = []
topic_index = 0
for topic in docs_per_topic_by_score:
    document_ids = [echoe.fileids()[k] for (k,v) in topic]
    topic_groups.append(document_ids)

In [15]:
topic_groups[0]

['382.13.txt',
 '394.24.txt',
 '038.35.txt',
 '068.03.txt',
 '331.78.txt',
 '049B.13.txt',
 '164.03.txt',
 '338.15.txt',
 '186.09i.txt',
 '186.19e.txt',
 '048.54.txt',
 '310.82.txt',
 '402.biii.txt',
 '041A.17.txt',
 '402.bii.txt',
 '049B.11.txt',
 '049B.41.txt',
 '386x.d.txt',
 '144.06.txt',
 '038.04.txt',
 '382.09.txt',
 '331.10b.txt',
 '057.43.txt',
 '068.10.txt',
 '186.19f.txt',
 '144.09.txt',
 '182.02b.txt',
 '331.30b.txt',
 '063.10.txt',
 '068.02.txt',
 '049B.30b.txt',
 '283.06.txt',
 '331.27.txt',
 '402.bi.txt',
 '331.20.txt',
 '309.01.txt',
 '049B.25.txt',
 '331.22.txt',
 '164.20.txt',
 '018.40.txt',
 '283.05.txt',
 '144.01.txt',
 '144.05.txt',
 '331.72.txt',
 '049B.40.txt',
 '164.02.txt',
 '186.19j.txt',
 '048.24.txt',
 '144.04.txt',
 '331.55.txt',
 '048.01.txt',
 '382.15.txt',
 '216.01.txt',
 '069.10.txt',
 '068.04.txt',
 '056.12.txt',
 '338.18.txt',
 '309.24.txt',
 '032.18.txt',
 '283.07.txt',
 '343.17.txt',
 '331.21.txt',
 '331.82.txt',
 '331.53.txt',
 '056.27.txt',
 '331.0

Results differ every time we run the model, but the most distinct topic has typically tended to include a lot of saints' lives (see the topic words above: "gelamp" means "[it] happened," and we get a number of proper nouns.) For instance, last time I ran the LDiA model, topic 0 was topped by `310.12`, `310.16`, and `382.16`. Are these thematically related? Let's have a look:

In [16]:
print(echoe.raw('310.12.txt')[:200])

her onginnæð to sæcgæn be þam treowe þe ðeo rode wæs of iwroht þe ure drihten for alles moncynnes hælo on ðrowode hu hit ærest weaxæn ongan 
we iherden sæcgen þurh sumne wisne mon þæt moyses þa þa he 


In [17]:
print(echoe.raw('310.16.txt')[:200])

natiuitas sancte marie 

men ða leofeste wurðie we nu on andweardnysse þa gebyrdtide þære eadige femne sancte marie ðeo wæs godes kenninge ures drihtnes hælendes cristes 
and hiræ nome ireht læfdi oðð


In [18]:
print(echoe.raw('382.16.txt')[:200])

to sanctae michaheles mæssan 

men ða leofestan manaþ us and myngaþ seo ar and seo eadignes þæs hean and þæs halgan heahengles tid þæt we hwæthwugu be þære his eadgan gemynde secgen þe is on ealra ymb


The association seems fair: `310.12` is the legend of the cross, which is comparable to saints' lives; `310.16` is the Nativity of Mary, likewise a saint's life; and `382.16` is on the archangel Saint Michael, and is similar in narrative style to the saints' lives. The saints' lives form a modest minority within ECHOE, but LDiA has found them out.